# Store Sales コンペ用ノートブック

このノートブックは、Kaggle の Store Sales - Time Series Forecasting に対して、ローカルの `uv` 環境で実行できるベースライン、CV、改善版提出までをまとめたものです。

- ベースライン提出の確認
- ローカル CV の作成
- 改善版の提出
- 実験ログと可視化

## 必要ライブラリの準備

既存のローカルパッケージ `store_sales` を利用します。Notebook 実行時は `uv run` で開くか、プロジェクトルートを `sys.path` に追加してください。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from store_sales.core import (
    VALIDATION_START,
    baseline_predict,
    hierarchical_mean_predict,
    load_raw_data,
    make_holdout_split,
    rmsle,
)

ROOT, VALIDATION_START

## Notebook 本体の作成

まずはデータを読み込み、ベースラインと改善版の比較に必要な前処理を確認します。

In [ ]:
bundle = load_raw_data()
train = bundle.train.copy()
test = bundle.test.copy()

train.head()

In [ ]:
summary = {
    'train_shape': train.shape,
    'test_shape': test.shape,
    'train_range': (train['date'].min(), train['date'].max()),
    'test_range': (test['date'].min(), test['date'].max()),
    'store_count': train['store_nbr'].nunique(),
    'family_count': train['family'].nunique(),
}
summary

## セルの実行と出力確認

曜日別売上と販促フラグ別売上を簡単に確認します。

In [ ]:
plot_df = train.copy()
plot_df['dayofweek'] = plot_df['date'].dt.dayofweek
weekly = plot_df.groupby('dayofweek')['sales'].mean().reindex(range(7))

fig, ax = plt.subplots(figsize=(8, 4))
weekly.plot(kind='bar', ax=ax, color='#4c78a8')
ax.set_title('Average sales by day of week')
ax.set_xlabel('dayofweek')
ax.set_ylabel('sales')
plt.tight_layout()
plt.show()

In [ ]:
promo_df = train.copy()
promo_df['promo_flag'] = (promo_df['onpromotion'] > 0).astype(int)
promo = promo_df.groupby('promo_flag')['sales'].mean().reindex([0, 1])

fig, ax = plt.subplots(figsize=(6, 4))
promo.plot(kind='bar', ax=ax, color=['#f58518', '#54a24b'])
ax.set_title('Average sales by promotion flag')
ax.set_xlabel('promotion flag')
ax.set_ylabel('sales')
plt.tight_layout()
plt.show()

## ベースライン提出

まずは store-family 平均ベースの最小実装です。提出形式が通ることを確認する目的で使います。

In [ ]:
baseline_submission = baseline_predict(train, test)
baseline_submission.head()

In [ ]:
train_cv, valid_cv = make_holdout_split(train, VALIDATION_START)
baseline_cv_pred = baseline_predict(train_cv, valid_cv)
baseline_cv = rmsle(valid_cv['sales'], baseline_cv_pred['sales'])
baseline_cv

## ローカル CV の確立と改善版

曜日と販促を使った階層平均を試します。今回のワークスペースでは、この方法が CV と LB の両方で最も安定しました。

In [ ]:
train_cv, valid_cv = make_holdout_split(train, VALIDATION_START)
valid_pred = hierarchical_mean_predict(train_cv, valid_cv)['sales']
valid_score = rmsle(valid_cv['sales'], valid_pred)
valid_score

In [ ]:
final_submission = hierarchical_mean_predict(train, test)
final_submission.head()

## 実験管理と可視化

実験ログは `experiments/runs.csv` に保存済みです。CV と LB の散布図は `reports/cv_lb_scatter.png` にあります。Notebook からでも確認できます。

In [ ]:
runs = pd.read_csv(ROOT / 'experiments' / 'runs.csv')
runs

## .ipynb 形式で保存

このファイル自体が `.ipynb` 形式の保存済み Notebook です。Kaggle Notebook として使う場合は、必要に応じてこの内容を新規 Notebook に貼り付けて保存します。

## 保存済み Notebook の再読み込み確認

Notebook ファイルは `notebooks/store_sales_competition.ipynb` に保存されています。再度開いてもセル構成と内容を保持できます。